Dependências
- Instala o kagglehub, caso necessário
- Carrega bibliotecas necessárias para execução

In [24]:
!pip install kagglehub[pandas-datasets]

In [25]:
# Dataset
import kagglehub
from kagglehub import KaggleDatasetAdapter
from typing import Tuple

# Pre-Processamento
import os
from pathlib import Path
from PIL import Image
from keras.preprocessing import image
import numpy as np
from keras.applications.imagenet_utils import preprocess_input
import random

# Train Test
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model
import numpy as np
from sklearn.preprocessing import LabelEncoder

Carregar Dataset
- Primeiramente, baixa o dataset na maquina pelo kagglehub

In [26]:
file_path = "HAM10000_metadata.csv"

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "kmader/skin-cancer-mnist-ham10000",
  file_path,
)
root = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

print(df.head())

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
     lesion_id      image_id   dx dx_type   age   sex localization
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear


In [27]:
labels = dict()

for i, l in  enumerate(df['dx'].unique()):
    labels[l] = i

In [28]:
import torch
from torch.utils.data import DataLoader, Dataset
class HAMDataset(Dataset):
    def __init__(self, dataframe, root, transform=None):
        # Fazer cópia e resetar índice
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.root = root
        self.transform = transform

        # Pré-processar: criar listas de caminhos e labels
        self.img_paths = []
        self.img_labels = []

        print("Pré-processando dataset...")
        for i in range(len(self.dataframe)):
            row = self.dataframe.iloc[i]
            img_name = row['image_id'] + '.jpg'
            diagnosis = row['dx']

            # Tentar part_1
            img_path = os.path.join(root, 'HAM10000_images_part_1', img_name)
            if not os.path.exists(img_path):
                img_path = os.path.join(root, 'HAM10000_images_part_2', img_name)

            if os.path.exists(img_path):
                self.img_paths.append(img_path)
                self.img_labels.append(labels[diagnosis])
            else:
                print(f"  Aviso: Imagem {img_name} não encontrada, pulando...")

        print(f"Dataset preparado: {len(self.img_paths)} imagens válidas de {len(dataframe)} totais")

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        # idx já deve ser inteiro do DataLoader
        # Mas vamos garantir
        if torch.is_tensor(idx):
            idx = idx.item()

        # Acessar listas pré-processadas
        img_path = self.img_paths[idx]
        label = self.img_labels[idx]

        # Carregar imagem
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

In [29]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

# ====================== TRANSFORMS ======================
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['dx'],
    random_state=42
)

print(f"Treino: {len(train_df)} imagens | Teste: {len(test_df)} imagens")
print(f"Distribuição de classes no treino:\n{train_df['dx'].value_counts()}\n")

train_dataset = HAMDataset(dataframe=train_df, root=root, transform=train_transform)
test_dataset  = HAMDataset(dataframe=test_df,  root=root, transform=test_transform)

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

Treino: 8012 imagens | Teste: 2003 imagens
Distribuição de classes no treino:
dx
nv       5364
mel       890
bkl       879
bcc       411
akiec     262
vasc      114
df         92
Name: count, dtype: int64

Pré-processando dataset...
Dataset preparado: 8012 imagens válidas de 8012 totais
Pré-processando dataset...
Dataset preparado: 2003 imagens válidas de 2003 totais


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [30]:
# 1 Batch de exemplo
print(f"Tamanho dataset: {len(dataset)}")
print(f"Total batches: {len(dataloader)}")

for batch_idx, (images, labels) in enumerate(dataloader):
    print(f"\nBatch {batch_idx}:")
    print(f"  Images shape: {images.shape}")
    print(f"  Labels shape: {labels.shape}")
    print(f"  Labels: {labels}")

    print(f"  First image: min: {images[0].min():.3f}, max: {images[0].max():.3f}")

    if batch_idx == 0:
        break

Tamanho dataset: 10015
Total batches: 313


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



Batch 0:
  Images shape: torch.Size([32, 3, 256, 256])
  Labels shape: torch.Size([32])
  Labels: tensor([1, 1, 1, 1, 6, 4, 1, 1, 3, 1, 1, 0, 3, 1, 1, 1, 4, 1, 6, 1, 1, 0, 1, 5,
        1, 1, 1, 1, 1, 0, 1, 3])
  First image: min: -1.423, max: 2.180


In [31]:
# Função treino
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(dataloader):
        # GPU
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # Estatisticas
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        if batch_idx % 50 == 0:
            print(f'  Batch {batch_idx}/{len(dataloader)} - Loss: {loss.item():.4f}')

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [32]:
from torchvision import models
import torch.nn as nn
import copy

# Lista de modelos para testar
lst_models_config = [
    {'name': 'VGG16', 'class': models.vgg16, 'pretrained': True},
    {'name': 'ResNet18', 'class': models.resnet18, 'pretrained': True},
    {'name': 'ResNet50', 'class': models.resnet50, 'pretrained': True}
]

lst_num_epochs = [3, 5, 10]

In [33]:
# Treinamento
for model_config in lst_models_config:
    model_name = model_config['name']
    print(f"\n{'='*60}")
    print(f"EXPERIMENTO COM: {model_name}")
    print(f"{'='*60}")

    for num_epochs in lst_num_epochs:
        print(f"\n{'='*40}")
        print(f"Treinando {model_name} por {num_epochs} epochs")
        print(f"{'='*40}")

        # 1. CRIAR NOVO MODELO PARA CADA EXPERIMENTO
        model = model_config['class'](pretrained=model_config['pretrained'])

        # 2. MODIFICAR PARA SEU NÚMERO DE CLASSES
        num_classes = len(df['dx'].unique())

        if 'vgg' in model_name.lower():
            # Para VGG
            for param in model.features.parameters():
                param.requires_grad = False
            model.classifier[6] = nn.Linear(4096, num_classes)
        else:
            # Para ResNet
            for param in model.parameters():
                param.requires_grad = False
            model.fc = nn.Linear(model.fc.in_features, num_classes)

        # 3. MOVER PARA GPU
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = model.to(device)

        # 4. NOVO OTIMIZADOR PARA CADA EXPERIMENTO
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

        # 5. TREINAMENTO
        for epoch in range(num_epochs):
            print(f"\nEpoch {epoch+1}/{num_epochs}")
            loss, acc = train_one_epoch(model, train_dataloader, criterion, optimizer, device)
            print(f"  Loss: {loss:.4f}, Accuracy: {acc:.2%}")

        # 6. TESTE COMPLETO (não apenas um batch)
        print("\nTestando modelo treinado...")
        model.eval()

        total_correct = 0
        total_samples = 0

        with torch.no_grad():
            for images, labels in test_dataloader:  # Dataloader separado para teste!
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                _, predicted = torch.max(outputs, 1)

                total_correct += (predicted == labels).sum().item()
                total_samples += len(labels)

        accuracy = total_correct / total_samples
        print(f"Accuracy total no teste: {accuracy:.2%}")
        print(f"Total de amostras testadas: {total_samples}")

        # 7. SALVAR RESULTADOS (opcional)
        results = {
            'model': model_name,
            'epochs': num_epochs,
            'accuracy': accuracy,
            'total_samples': total_samples
        }
        print(f"Resultados: {results}")


EXPERIMENTO COM: VGG16

Treinando VGG16 por 3 epochs


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Epoch 1/3
  Batch 0/251 - Loss: 2.0690
  Batch 50/251 - Loss: 1.1901
  Batch 100/251 - Loss: 1.3708
  Batch 150/251 - Loss: 1.3062
  Batch 200/251 - Loss: 1.2757
  Batch 250/251 - Loss: 2.2864
  Loss: 1.1471, Accuracy: 65.19%

Epoch 2/3
  Batch 0/251 - Loss: 1.8063
  Batch 50/251 - Loss: 0.9102
  Batch 100/251 - Loss: 0.6841
  Batch 150/251 - Loss: 0.9966
  Batch 200/251 - Loss: 1.0396
  Batch 250/251 - Loss: 1.4030
  Loss: 1.0844, Accuracy: 65.75%

Epoch 3/3
  Batch 0/251 - Loss: 1.0955
  Batch 50/251 - Loss: 0.6656
  Batch 100/251 - Loss: 0.7231
  Batch 150/251 - Loss: 0.5460
  Batch 200/251 - Loss: 1.8737
  Batch 250/251 - Loss: 0.8272
  Loss: 1.0437, Accuracy: 67.57%

Testando modelo treinado...
Accuracy total no teste: 70.19%
Total de amostras testadas: 2003
Resultados: {'model': 'VGG16', 'epochs': 3, 'accuracy': 0.7019470793809286, 'total_samples': 2003}

Treinando VGG16 por 5 epochs

Epoch 1/5
  Batch 0/251 - Loss: 2.3232
  Batch 50/251 - Loss: 1.4130
  Batch 100/251 - Loss: 0.

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


  Batch 0/251 - Loss: 3.0633
  Batch 50/251 - Loss: 1.0901
  Batch 100/251 - Loss: 1.0103
  Batch 150/251 - Loss: 0.7043
  Batch 200/251 - Loss: 0.6471
  Batch 250/251 - Loss: 0.9398
  Loss: 0.9332, Accuracy: 68.25%

Epoch 2/3
  Batch 0/251 - Loss: 0.6420
  Batch 50/251 - Loss: 0.8100
  Batch 100/251 - Loss: 0.5487
  Batch 150/251 - Loss: 0.5978
  Batch 200/251 - Loss: 0.6408
  Batch 250/251 - Loss: 0.5243
  Loss: 0.7514, Accuracy: 73.33%

Epoch 3/3
  Batch 0/251 - Loss: 0.5777
  Batch 50/251 - Loss: 0.7700
  Batch 100/251 - Loss: 0.6890
  Batch 150/251 - Loss: 0.9305
  Batch 200/251 - Loss: 0.8570
  Batch 250/251 - Loss: 0.7015
  Loss: 0.7141, Accuracy: 74.44%

Testando modelo treinado...
Accuracy total no teste: 76.19%
Total de amostras testadas: 2003
Resultados: {'model': 'ResNet18', 'epochs': 3, 'accuracy': 0.7618572141787319, 'total_samples': 2003}

Treinando ResNet18 por 5 epochs

Epoch 1/5
  Batch 0/251 - Loss: 2.4079
  Batch 50/251 - Loss: 0.9260
  Batch 100/251 - Loss: 0.9208


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 121MB/s]



Epoch 1/3
  Batch 0/251 - Loss: 1.7209
  Batch 50/251 - Loss: 0.8655
  Batch 100/251 - Loss: 0.9251
  Batch 150/251 - Loss: 0.6987
  Batch 200/251 - Loss: 0.6640
  Batch 250/251 - Loss: 0.8599
  Loss: 0.8895, Accuracy: 69.63%

Epoch 2/3
  Batch 0/251 - Loss: 0.7195
  Batch 50/251 - Loss: 0.6877
  Batch 100/251 - Loss: 0.7682
  Batch 150/251 - Loss: 0.9083
  Batch 200/251 - Loss: 0.8990
  Batch 250/251 - Loss: 0.3836
  Loss: 0.7294, Accuracy: 73.38%

Epoch 3/3
  Batch 0/251 - Loss: 0.6813
  Batch 50/251 - Loss: 0.7336
  Batch 100/251 - Loss: 0.6739
  Batch 150/251 - Loss: 1.0231
  Batch 200/251 - Loss: 0.6903
  Batch 250/251 - Loss: 1.2493
  Loss: 0.7021, Accuracy: 74.40%

Testando modelo treinado...
Accuracy total no teste: 74.79%
Total de amostras testadas: 2003
Resultados: {'model': 'ResNet50', 'epochs': 3, 'accuracy': 0.7478781827259111, 'total_samples': 2003}

Treinando ResNet50 por 5 epochs

Epoch 1/5
  Batch 0/251 - Loss: 2.1097
  Batch 50/251 - Loss: 0.8196
  Batch 100/251 - Lo